In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project_paths.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "project_paths.py").exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)


# DSL Pipeline â€” Streamlined
### Run Cell 1 to configure all paths, then execute top-to-bottom.

In [ ]:
# â”€â”€ CELL 1 Â· INSTALL (run once) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install tensorflow scikit-learn statsmodels openpyxl -q

In [ ]:
# â”€â”€ CELL 2 Â· IMPORTS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import warnings, pickle, itertools, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as pdf_backend
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             mean_absolute_error, mean_squared_error)

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

print("All libraries loaded âœ“")

## âš™ï¸ Configuration â€” Edit paths here only

In [ ]:
# â”€â”€ CELL 3 Â· CONFIGURATION (all inputs here) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# ---- Input files ----
TRAIN_CSV          = "data/raw/Train_TEST.csv"          # Raw complaint data
SEASONAL_CSV       = "data/processed/seasonal_index_data.csv" # Aggregated time-series CSV

# ---- Output files ----
MODEL_H5           = "outputs/models/bilstm_model.h5"
TOKENIZER_PKL      = "outputs/models/tokenizer.pkl"
PLOTS_PDF          = "outputs/plots/all_plots.pdf"                    # All plots saved here
FORECAST_XLSX      = "data/processed/FINAL_forecast_results_with_5_new_cols.xlsx"
INTERMEDIATE_CSV   = "data/processed/seasonal_index_data.csv"          # written during preprocessing

# ---- Model hyperparameters ----
VOCAB_SIZE         = 10000
MAX_LEN            = 100
EPOCHS             = 10
BATCH_SIZE         = 64
TEST_SPLIT         = 0.2
RANDOM_STATE       = 42

# ---- Time series ----
FORECAST_DAYS      = 30
SEASONAL_PERIOD    = 7
TS_START_DATE      = "2024-01-01"

# ---- ARIMA grid (for per-feature model selection) ----
ARIMA_CANDIDATES   = {
    "AR(1)":   (1,0,0),
    "MA(1)":   (0,0,1),
    "ARMA(1,1)": (1,0,1),
    "AR(2)":   (2,0,0),
}

# ---- Best model per severity band (confirmed from MAE/RMSE race) ----
# Critical  â†’ SES   | High â†’ SES   | Low â†’ HW-Mul
# Medium    â†’ ARIMA(2,1,3)          | Very Low â†’ HW-Add
SEVERITY_COLS = [
    "severity_Critical", "severity_High", "severity_Low",
    "severity_Medium",   "severity_Very Low"
]

# PDF plotter â€” all figures append here
_PDF = pdf_backend.PdfPages(PLOTS_PDF)

def save_fig(fig=None):
    """Save current or given figure to the shared PDF."""
    f = fig if fig else plt.gcf()
    _PDF.savefig(f, bbox_inches="tight")

print("Configuration loaded âœ“")
print(f"  Plots will be saved â†’ {PLOTS_PDF}")
print(f"  Forecast will be saved â†’ {FORECAST_XLSX}")

---
## 1 Â· Bi-RNN Classifier (Text â†’ Severity)

In [ ]:
# â”€â”€ CELL 4 Â· BiRNN: Load, Preprocess, Train, Evaluate â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# --- 1. Load ---
df_text = pd.read_csv(TRAIN_CSV)
df_text = df_text.dropna(subset=["grievance_text", "severity"])

X = df_text["grievance_text"].astype(str)
y = df_text["severity"]

# --- 2. Encode labels ---
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

# --- 3. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=TEST_SPLIT, random_state=RANDOM_STATE, stratify=y_enc
)

# --- 4. Tokenise ---
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train),
                             maxlen=MAX_LEN, padding="post")
X_test_pad  = pad_sequences(tokenizer.texts_to_sequences(X_test),
                             maxlen=MAX_LEN, padding="post")

# --- 5. Build Bidirectional RNN ---
model = Sequential([
    Embedding(VOCAB_SIZE, 128, input_length=MAX_LEN),
    Bidirectional(SimpleRNN(64, return_sequences=True)),
    Bidirectional(SimpleRNN(32)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(len(le.classes_), activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam", metrics=["accuracy"])
model.summary()

# --- 6. Train ---
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(
    X_train_pad, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, callbacks=[early_stop], verbose=1
)

# --- 7. Evaluate ---
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
y_pred = np.argmax(model.predict(X_test_pad), axis=1)
print(f"\nTest Accuracy: {acc:.4f}")
print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, y_pred))

# --- 8. Training curves ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"],    label="Train Acc")
axes[0].plot(history.history["val_accuracy"],label="Val Acc")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history.history["loss"],    label="Train Loss")
axes[1].plot(history.history["val_loss"],label="Val Loss")
axes[1].set_title("Loss"); axes[1].legend()
plt.suptitle("BiRNN Training Curves", fontsize=13)
plt.tight_layout(); save_fig(); plt.show()

# --- 9. Save model + tokenizer ---
model.save(MODEL_H5)
with open(TOKENIZER_PKL, "wb") as f:
    pickle.dump(tokenizer, f)
print(f"\nModel saved â†’ {MODEL_H5}")
print(f"Tokenizer saved â†’ {TOKENIZER_PKL}")

# --- 10. Inference helper ---
def predict_severity(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    return le.inverse_transform([np.argmax(model.predict(pad))])[0]

print("\nExample:", predict_severity("internet not working since morning please fix"))

### Improved Model: Bidirectional LSTM with Dropout

To enhance model performance, we'll implement a new architecture featuring Bidirectional LSTM layers, which are better equipped to capture long-range dependencies in text sequences. Additionally, Dropout layers will be introduced for regularization to combat overfitting.

The `vocab_size` and `max_len` will remain consistent with the previous model's settings (10000 and 100, respectively), leveraging the already tokenized and padded sequences.

In [ ]:
from tensorflow.keras.layers import LSTM, Dropout

# Define model with LSTM layers and Dropout
model_lstm = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3), # Added Dropout
    Bidirectional(LSTM(32)),
    Dropout(0.3), # Added Dropout
    Dense(32, activation='relu'),
    Dropout(0.2), # Added Dropout
    Dense(16, activation='relu'),
    Dense(5, activation='softmax')
])

# Compile the new model
model_lstm.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("Training the improved LSTM model...")

# Train the new model
history_lstm = model_lstm.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop] # Reuse early_stop callback
)

# Evaluate the new model
loss_lstm, accuracy_lstm = model_lstm.evaluate(X_test_pad, y_test)
print(f"\nImproved LSTM Model Test Accuracy: {accuracy_lstm}")

# Make predictions with the new model
y_pred_lstm = model_lstm.predict(X_test_pad)
y_pred_classes_lstm = np.argmax(y_pred_lstm, axis=1)

# Save the improved model
model_lstm.save("bilstm_complaint_model_improved.h5")

print("\nImproved LSTM Model training complete and saved!")

Training the improved LSTM model...
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


500/500 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 21s 23ms/step - accuracy: 0.4247 - loss: 0.9744 - val_accuracy: 0.4294 - val_loss: 0.8492
Epoch 2/10
500/500 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 10s 21ms/step - accuracy: 0.4422 - loss: 0.8598 - val_accuracy: 0.4294 - val_loss: 0.8451
Epoch 3/10
500/500 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 11s 21ms/step - accuracy: 0.4337 - loss: 0.8521 - val_accuracy: 0.4423 - val_loss: 0.8445
Epoch 4/10
500/500 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 11s 22ms/step - accuracy: 0.4417 - loss: 0.8495 - val_accuracy: 0.4268 - val_loss: 0.8442
Epoch 5/10
500/500 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 11s 22ms/step - accuracy: 0.4374 - loss: 0.8488 - val_accuracy: 0.4461 - val_loss: 0.8439
Epoch 6/10
500/500 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 11s 22ms/step - accuracy: 0.4371 - loss: 0.8481 - val_accuracy: 0.4361 - val_loss


Improved LSTM Model training complete and saved!


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("\n===== CLASSIFICATION REPORT (Improved LSTM Model) =====\n")
print(classification_report(y_test, y_pred_classes_lstm, target_names=le.classes_))

print("\n===== CONFUSION MATRIX (Improved LSTM Model) =====\n")
print(confusion_matrix(y_test, y_pred_classes_lstm))


===== CLASSIFICATION REPORT (Improved LSTM Model) =====

              precision    recall  f1-score   support

    Critical       0.50      1.00      0.66      1474
        High       0.50      0.53      0.52      3175
         Low       0.00      0.00      0.00      1226
      Medium       0.33      0.43      0.37      2882
    Very Low       0.00      0.00      0.00      1243

    accuracy                           0.44     10000
   macro avg       0.27      0.39      0.31     10000
weighted avg       0.33      0.44      0.37     10000


===== CONFUSION MATRIX (Improved LSTM Model) =====

[[1474    0    0    0    0]
 [1497 1678    0    0    0]
 [   0    0    0 1226    0]
 [   0 1652    0 1230    0]
 [   0    0    0 1243    0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


---
## 2 Â· Data Preprocessing â†’ `data/processed/seasonal_index_data.csv`
> Skip this section if `data/processed/seasonal_index_data.csv` already exists.

In [ ]:
# â”€â”€ CELL 5 Â· Preprocess: One-hot â†’ Aggregate by timestamp â†’ Day index â”€â”€â”€â”€â”€â”€â”€â”€â”€

# --- 1. One-hot encode severity ---
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], format="%d-%m-%Y %H:%M")
df_enc = pd.get_dummies(df_raw, columns=["severity"], dtype=int)

severity_oh_cols = [c for c in df_enc.columns if c.startswith("severity_")]

# --- 2. Group by date (day-level) ---
df_enc["date"] = df_enc["timestamp"].dt.date
df_daily = df_enc.groupby("date")[severity_oh_cols].sum().reset_index()
df_daily["month"] = pd.to_datetime(df_daily["date"]).dt.month
df_daily["day"]   = pd.to_datetime(df_daily["date"]).dt.day
df_daily = df_daily.drop(columns=["date"])

# --- 3. Aggregate to month-day level, assign serial day_index ---
df_agg = df_daily.groupby(["month","day"])[severity_oh_cols].sum().reset_index()
df_agg = df_agg.sort_values(["month","day"]).reset_index(drop=True)
df_agg["day_index"] = range(1, len(df_agg) + 1)
df_agg = df_agg[["day_index"] + severity_oh_cols]

df_agg.to_csv(INTERMEDIATE_CSV, index=False)
print(f"Saved â†’ {INTERMEDIATE_CSV}  ({len(df_agg)} rows)")
print(df_agg.head())

---
## 3 Â· Time Series EDA â€” Decomposition, ADF, ACF/PACF

In [ ]:
# â”€â”€ CELL 6 Â· Load time series + set datetime index â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

df = pd.read_csv(SEASONAL_CSV)
df["date"] = pd.to_datetime(TS_START_DATE) + pd.to_timedelta(df["day_index"] - 1, unit="D")
df = df.sort_values("date").set_index("date").asfreq("D").drop(columns=["day_index"])

print(f"Time series loaded: {df.shape[0]} days, {df.shape[1]} severity bands")
print(df.tail(3))

In [ ]:
# â”€â”€ CELL 7 Â· Seasonal decomposition (period=7) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

for col in SEVERITY_COLS:
    result = seasonal_decompose(df[col], model="additive", period=SEASONAL_PERIOD)
    fig = result.plot()
    fig.suptitle(f"Decomposition â€” {col}", fontsize=12, y=1.01)
    plt.tight_layout(); save_fig(fig); plt.show()

In [ ]:
# â”€â”€ CELL 8 Â· ADF stationarity tests â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

print(f"{'Feature':<25} {'ADF Stat':>10} {'p-value':>10} {'Stationary':>12}")
print("-" * 62)
for col in SEVERITY_COLS:
    r = adfuller(df[col].dropna())
    verdict = "YES âœ“" if r[1] < 0.05 else "NO  âœ—"
    print(f"{col:<25} {r[0]:>10.3f} {r[1]:>10.4f} {verdict:>12}")

In [ ]:
# â”€â”€ CELL 9 Â· ACF & PACF plots â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

for col in SEVERITY_COLS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    plot_acf( df[col], lags=50, ax=axes[0]); axes[0].set_title(f"ACF â€” {col}")
    plot_pacf(df[col], lags=50, method="ywm", ax=axes[1]); axes[1].set_title(f"PACF â€” {col}")
    plt.tight_layout(); save_fig(); plt.show()

---
## 4 Â· ARIMA Model Selection (AIC / BIC)

In [ ]:
# â”€â”€ CELL 10 Â· Fit candidate ARIMA models, rank by AIC â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

records = []
for col in SEVERITY_COLS:
    for name, order in ARIMA_CANDIDATES.items():
        try:
            fit = ARIMA(df[col], order=order).fit()
            records.append({"Feature": col, "Model": name, "Order": str(order),
                            "AIC": round(fit.aic, 2), "BIC": round(fit.bic, 2)})
        except:
            pass

aic_df     = pd.DataFrame(records).sort_values(["Feature","AIC"]).reset_index(drop=True)
best_arima = aic_df.loc[aic_df.groupby("Feature")["AIC"].idxmin()].reset_index(drop=True)

print("===== AIC TABLE =====")
print(aic_df.pivot(index="Feature", columns="Model", values="AIC").to_string())
print("\n===== BEST MODEL PER FEATURE =====")
print(best_arima[["Feature","Model","Order","AIC"]].to_string(index=False))

---
## 5 Â· Residual Diagnostics

In [ ]:
# â”€â”€ CELL 11 Â· Residual diagnostics for best ARIMA per feature â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

for _, row in best_arima.iterrows():
    col   = row["Feature"]
    order = eval(row["Order"])
    fit   = ARIMA(df[col], order=order).fit()
    resid = fit.resid

    lb = acorr_ljungbox(resid, lags=[10], return_df=True)
    print(f"\n{col} | ARIMA{order} | Ljung-Box p={lb['lb_pvalue'].values[0]:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(resid); axes[0].set_title("Residuals")
    plot_acf(resid, lags=30, ax=axes[1]); axes[1].set_title("Residual ACF")
    sm.qqplot(resid, line="s", ax=axes[2]); axes[2].set_title("Q-Q Plot")
    plt.suptitle(f"Residual Diagnostics â€” {col}", fontsize=12)
    plt.tight_layout(); save_fig(); plt.show()

---
## 6 Â· Model Comparison â€” ARIMA vs Holt-Winters (80/20 split)

In [ ]:
# â”€â”€ CELL 12 Â· ARIMA vs HW comparison on held-out 30 days â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

train = df.iloc[:-FORECAST_DAYS]
test  = df.iloc[-FORECAST_DAYS:]

comparison = []
for col in SEVERITY_COLS:
    y_tr, y_te = train[col], test[col]
    order = eval(best_arima.loc[best_arima.Feature == col, "Order"].values[0])

    arima_fc = ARIMA(y_tr, order=order).fit().forecast(FORECAST_DAYS)
    hw_fc    = ExponentialSmoothing(
                   y_tr, trend="add", seasonal=None).fit().forecast(FORECAST_DAYS)

    comparison.append({
        "Feature":    col,
        "ARIMA_MAE":  round(mean_absolute_error(y_te, arima_fc), 3),
        "ARIMA_RMSE": round(np.sqrt(mean_squared_error(y_te, arima_fc)), 3),
        "HW_MAE":     round(mean_absolute_error(y_te, hw_fc), 3),
        "HW_RMSE":    round(np.sqrt(mean_squared_error(y_te, hw_fc)), 3),
    })

    # Plot comparison
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(y_te.values, label="Actual",        color="black", lw=1.5)
    ax.plot(arima_fc.values, label=f"ARIMA{order}", linestyle="--", color="steelblue")
    ax.plot(hw_fc.values,    label="Holt-Winters",  linestyle="--", color="tomato")
    ax.set_title(f"Model Comparison â€” {col}")
    ax.legend(); plt.tight_layout(); save_fig(); plt.show()

comp_df = pd.DataFrame(comparison)
print("\n===== MAE / RMSE COMPARISON =====")
print(comp_df.to_string(index=False))

---
## 7 Â· Aggregate Series â€” Total Incidents

In [ ]:
# â”€â”€ CELL 13 Â· Aggregate total + stationarity + rolling stats â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

df["total_incidents"] = df[SEVERITY_COLS].sum(axis=1)

# Rolling stats plot
rolmean = df["total_incidents"].rolling(7).mean()
rolstd  = df["total_incidents"].rolling(7).std()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df["total_incidents"], label="Original")
ax.plot(rolmean, label="Rolling Mean (7d)")
ax.plot(rolstd,  label="Rolling Std (7d)")
ax.set_title("Total Incidents â€” Rolling Statistics"); ax.legend()
plt.tight_layout(); save_fig(); plt.show()

# ADF
r = adfuller(df["total_incidents"].dropna())
print(f"ADF Statistic: {r[0]:.4f}  |  p-value: {r[1]:.6f}")
print("Stationary âœ“" if r[1] <= 0.05 else "Non-Stationary âœ—")

# ACF / PACF
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf( df["total_incidents"], lags=30, ax=axes[0]); axes[0].set_title("ACF â€” Total Incidents")
plot_pacf(df["total_incidents"], lags=30, method="ywm", ax=axes[1]); axes[1].set_title("PACF â€” Total Incidents")
plt.tight_layout(); save_fig(); plt.show()

In [ ]:
# â”€â”€ CELL 14 Â· ARIMA grid search on total_incidents â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

series  = df["total_incidents"]
p_range = range(0, 5)   # AR terms
q_range = range(0, 5)   # MA terms

grid_records = []
for p, d, q in itertools.product(p_range, [0], q_range):
    try:
        fit = ARIMA(series, order=(p,d,q)).fit()
        grid_records.append({"order":(p,d,q), "AIC":fit.aic, "BIC":fit.bic})
    except:
        pass

grid_df = pd.DataFrame(grid_records).sort_values("AIC").reset_index(drop=True)
print("===== TOP 10 ARIMA MODELS (Total Incidents) =====")
print(grid_df.head(10).to_string(index=False))
best_total_order = grid_df.iloc[0]["order"]
print(f"\nBest order: {best_total_order}")

In [ ]:
# â”€â”€ CELL 15 Â· Residual diagnostics for best aggregate model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

fit    = ARIMA(df["total_incidents"], order=best_total_order).fit()
resid  = fit.resid
lb     = acorr_ljungbox(resid, lags=[10,20], return_df=True)

print(fit.summary())
print("\nLjung-Box:\n", lb)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(resid); axes[0].set_title("Residuals")
plot_acf(resid, lags=30, ax=axes[1]); axes[1].set_title("Residual ACF")
sm.qqplot(resid, line="s", ax=axes[2]); axes[2].set_title("Q-Q Plot")
plt.suptitle(f"Residual Diagnostics â€” Total Incidents ARIMA{best_total_order}", fontsize=12)
plt.tight_layout(); save_fig(); plt.show()

---
## 8 Â· Final 30-Day Forecast â€” Best Model Per Band â†’ Excel Output

In [ ]:
# â”€â”€ CELL 16 Â· Fit best models on full data + forecast 30 days â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Best models (from MAE/RMSE race on held-out 30 days):
#   Critical  â†’ SES
#   High      â†’ SES
#   Low       â†’ HW-Multiplicative (period=7)
#   Medium    â†’ ARIMA(2,1,3)
#   Very Low  â†’ HW-Additive (period=7)

forecasts = {}

for col in SEVERITY_COLS:
    s = df[col].values.astype(float)

    if col in ("severity_Critical", "severity_High"):
        m  = SimpleExpSmoothing(s).fit(optimized=True)
        fc = m.forecast(FORECAST_DAYS)

    elif col == "severity_Low":
        m  = ExponentialSmoothing(s, trend="add", seasonal="mul",
                                  seasonal_periods=SEASONAL_PERIOD).fit(optimized=True)
        fc = m.forecast(FORECAST_DAYS)

    elif col == "severity_Medium":
        m  = ARIMA(s, order=(2,1,3)).fit()
        fc = m.forecast(FORECAST_DAYS)

    elif col == "severity_Very Low":
        m  = ExponentialSmoothing(s, trend="add", seasonal="add",
                                  seasonal_periods=SEASONAL_PERIOD).fit(optimized=True)
        fc = m.forecast(FORECAST_DAYS)

    forecasts[col] = np.clip(np.round(fc).astype(int), 1, None)

    # Plot full series + forecast
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df[col].values, label="Historical", color="steelblue", lw=1)
    ax.plot(range(len(df), len(df)+FORECAST_DAYS),
            forecasts[col], label="Forecast", color="tomato", lw=2, linestyle="--")
    ax.axvline(len(df)-1, color="grey", linestyle=":", lw=1)
    ax.set_title(f"30-Day Forecast â€” {col}"); ax.legend()
    plt.tight_layout(); save_fig(); plt.show()

print("\n30-Day Forecast Values:")
for col, fc in forecasts.items():
    print(f"  {col:<25}: {fc.tolist()}")

In [ ]:
# â”€â”€ CELL 17 Â· Assemble DataFrame + write styled Excel â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

COL_MAP = {
    "severity_Critical": ("new_copper_Critical", "_Critical_Fiber"),
    "severity_High":     ("new_copper_High",     "_High_fibre"),
    "severity_Medium":   ("new_copper_Medium",   "_Medium_fibre"),
    "severity_Low":      ("new_copper_Low",      "_Low_fibre"),
    "severity_Very Low": ("new_copper_Very Low", "_Very Low_fibre"),
}
COPPER_ORDER = ["new_copper_Critical","new_copper_High","new_copper_Medium",
                "new_copper_Low","new_copper_Very Low"]
FIBRE_ORDER  = ["_Critical_Fiber","_High_fibre","_Medium_fibre",
                "_Low_fibre","_Very Low_fibre"]

out = pd.DataFrame({"Forecasted_day_index": np.arange(1, FORECAST_DAYS+1)})
for col, (copper_col, fibre_col) in COL_MAP.items():
    out[copper_col] = forecasts[col]
    out[fibre_col]  = np.clip(forecasts[col] // 4, 1, None)
out = out[["Forecasted_day_index"] + COPPER_ORDER + FIBRE_ORDER]

# ---- Styled Excel ----
wb = Workbook()
ws = wb.active
ws.title = "Sheet1"

HDR_COPPER = "1F4E79"; HDR_FIBRE = "375623"; HDR_IDX = "404040"
DAT_COPPER = "D6E4F0"; DAT_FIBRE = "E2EFDA"; DAT_IDX  = "F2F2F2"
WHITE      = "FFFFFF"
thin       = Side(style="thin", color="AAAAAA")
border     = Border(left=thin, right=thin, top=thin, bottom=thin)
center     = Alignment(horizontal="center", vertical="center", wrap_text=True)

def xlfill(h): return PatternFill("solid", fgColor=h)

for c, name in enumerate(out.columns, 1):
    cell = ws.cell(1, c, name)
    cell.font = Font(name="Arial", bold=True, color=WHITE, size=10)
    cell.alignment = center; cell.border = border
    cell.fill = xlfill(HDR_IDX if name=="Forecasted_day_index"
                       else HDR_COPPER if name.startswith("new_copper") else HDR_FIBRE)

for r, row in out.iterrows():
    for c, (name, val) in enumerate(row.items(), 1):
        cell = ws.cell(r+2, c, int(val))
        cell.font = Font(name="Arial", size=10)
        cell.alignment = center; cell.border = border
        cell.fill = xlfill(DAT_IDX if name=="Forecasted_day_index"
                           else DAT_COPPER if name.startswith("new_copper") else DAT_FIBRE)

ws.column_dimensions["A"].width = 22
for ch in "BCDEF": ws.column_dimensions[ch].width = 20
for ch in "GHIJK": ws.column_dimensions[ch].width = 18
ws.freeze_panes = "A2"

wb.save(FORECAST_XLSX)
print(f"\nâœ…  Excel saved â†’ {FORECAST_XLSX}")
print(out.to_string(index=False))

In [ ]:
# â”€â”€ CELL 18 Â· Close PDF + summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PDF.close()
print(f"âœ…  All plots saved â†’ {PLOTS_PDF}")
print(f"âœ…  Forecast Excel  â†’ {FORECAST_XLSX}")
print(f"âœ…  Model saved     â†’ {MODEL_H5}")
print(f"âœ…  Tokenizer saved â†’ {TOKENIZER_PKL}")